# ACCIDENT @ CVPR: R(2+1)D Spatiotemporal Baseline

This notebook builds a clean baseline for the ACCIDENT @ CVPR competition using a spatiotemporal video backbone. The default model is `torchvision.models.video.r2plus1d_18`, which is a better first baseline for this task than a frame-only detector because the competition requires understanding when the collision happens, where it happens, and what type it is from a video clip.


## 1. Setup and Imports

Lightweight, reproducible, CUDA/AMP-aware. Falls back gracefully when optional video backends are missing.


In [ ]:
import gc
import json
import math
import os
import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models.video import r2plus1d_18
from torchvision.transforms import InterpolationMode

SEED = 42


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Running on: {DEVICE}")


In [ ]:
def optional_import(name: str):
    try:
        module = __import__(name)
        print(f"Loaded optional dependency: {name}")
        return module
    except Exception as exc:
        print(f"Optional dependency unavailable: {name} ({exc})")
        return None


cv2 = optional_import("cv2")
decord = optional_import("decord")
av = optional_import("av")


## 2. Data Preparation & Inspection

The notebook uses data colocated with the notebook (e.g., `sim_dataset`, `videos`, `test_metadata.csv`). `/kaggle/input` is not required, but if present it will be considered as a fallback. No archive extraction is performed; data is assumed to be already extracted.


In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "test_metadata.csv").exists():
            return candidate
    return start.resolve()

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_ROOTS = [PROJECT_ROOT]
if Path("/kaggle/input").exists():
    DATA_ROOTS.append(Path("/kaggle/input"))

WORKING_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else (PROJECT_ROOT / "outputs" / "accident-r2plus1d-baseline")
WORKING_ROOT.mkdir(parents=True, exist_ok=True)


def score_dir(root: Path) -> Tuple[int, Path]:
    files = {p.name.lower(): p for p in root.rglob("*") if p.is_file()}
    score = 0
    if "labels.csv" in files:
        score += 3
    if "test_metadata.csv" in files:
        score += 3
    if any("submission" in name for name in files):
        score += 2
    if any(name.endswith(".mp4") for name in files):
        score += 1
    return score, root


def select_comp_root() -> Path:
    candidates: List[Tuple[int, Path]] = []
    for base in DATA_ROOTS:
        for child in [base] + [p for p in base.iterdir() if p.is_dir()]:
            score, scored_root = score_dir(child)
            if score > 0:
                candidates.append((score, scored_root))
    if not candidates:
        return DATA_ROOTS[0]
    candidates.sort(key=lambda x: (-x[0], len(str(x[1]))))
    return candidates[0][1]


COMP_ROOT = select_comp_root()
print(f"Competition root: {COMP_ROOT}")


In [ ]:
def print_tree(root: Path, max_depth: int = 3, max_entries: int = 250) -> None:
    shown = 0
    for path in sorted(root.rglob("*")):
        depth = len(path.relative_to(root).parts)
        if depth > max_depth:
            continue
        indent = "  " * (depth - 1)
        suffix = "/" if path.is_dir() else ""
        print(f"{indent}{path.name}{suffix}")
        shown += 1
        if shown >= max_entries:
            print(f"... truncated after {max_entries} entries")
            break


print_tree(COMP_ROOT, max_depth=3, max_entries=200)


In [ ]:
def find_files(root: Path) -> Dict[str, Optional[Path]]:
    files = [p for p in root.rglob("*") if p.is_file()]

    def first_match(predicate):
        for path in files:
            if predicate(path):
                return path
        return None

    return {
        "train_labels": first_match(lambda p: p.name.lower() == "labels.csv"),
        "test_metadata": first_match(lambda p: p.name.lower() == "test_metadata.csv"),
        "sample_submission": first_match(lambda p: "sample" in p.name.lower() and "submission" in p.name.lower()),
        "any_submission": first_match(lambda p: "submission" in p.name.lower() and p.suffix.lower() == ".csv"),
        "annotation_classes": first_match(lambda p: p.name.lower() == "annotation_classes.yaml"),
    }


found_files = find_files(COMP_ROOT)
found_files


In [ ]:
def safe_read_table(path: Optional[Path], nrows: Optional[int] = None) -> Optional[pd.DataFrame]:
    if path is None or not path.exists():
        return None
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, nrows=nrows)
    if path.suffix.lower() in {".json", ".jsonl"}:
        return pd.read_json(path)
    warnings.warn(f"Unsupported table format: {path}")
    return None


train_df = safe_read_table(found_files["train_labels"])
test_df = safe_read_table(found_files["test_metadata"])
sample_submission_df = safe_read_table(found_files["sample_submission"])
if sample_submission_df is None:
    sample_submission_df = safe_read_table(found_files["any_submission"])

print("train_df shape:", None if train_df is None else train_df.shape)
print("test_df shape:", None if test_df is None else test_df.shape)
print("sample_submission_df shape:", None if sample_submission_df is None else sample_submission_df.shape)

if train_df is not None:
    display(train_df.head())
if test_df is not None:
    display(test_df.head())
if sample_submission_df is not None:
    display(sample_submission_df.head())


In [ ]:
def summarize_schema(train_df: Optional[pd.DataFrame], test_df: Optional[pd.DataFrame], sample_submission_df: Optional[pd.DataFrame]) -> Dict[str, Any]:
    train_cols = set(train_df.columns) if train_df is not None else set()
    test_cols = set(test_df.columns) if test_df is not None else set()
    sub_cols = list(sample_submission_df.columns) if sample_submission_df is not None else []
    summary = {
        "video_path_columns": [c for c in ["rgb_path", "video_path", "path", "filepath", "file_name"] if c in train_cols.union(test_cols)],
        "time_target_columns": [c for c in ["accident_time", "time", "timestamp", "event_time"] if c in train_cols],
        "spatial_target_columns": [c for c in ["center_x", "center_y", "x1", "y1", "x2", "y2"] if c in train_cols],
        "cause_target_columns": [c for c in ["type", "cause", "label", "class", "collision_type"] if c in train_cols],
        "train_columns": list(train_df.columns) if train_df is not None else [],
        "test_columns": list(test_df.columns) if test_df is not None else [],
        "submission_columns": sub_cols,
        "has_train": train_df is not None,
        "has_test": test_df is not None,
        "has_sample_submission": sample_submission_df is not None,
    }
    return summary


schema_summary = summarize_schema(train_df, test_df, sample_submission_df)
print(json.dumps(schema_summary, indent=2))


### Discovered Schema

Expected for ACCIDENT:
- training labels in `sim_dataset/labels.csv`
- test metadata in `test_metadata.csv`
- submission columns: `path, accident_time, center_x, center_y, type`


## 3. Problem Formulation

- `accident_time`: regression (normalized by duration during training)
- `center_x`, `center_y`: point regression in normalized image coordinates
- `type`: categorical classification if present

Heads are created only when the corresponding labels exist.


In [ ]:
def infer_task_config(train_df: Optional[pd.DataFrame], test_df: Optional[pd.DataFrame], sample_submission_df: Optional[pd.DataFrame]) -> Dict[str, Any]:
    task = {"video_col": None, "time_col": None, "spatial_cols": [], "bbox_cols": [], "cause_col": None, "cause_classes": [], "group_col": None, "submission_columns": list(sample_submission_df.columns) if sample_submission_df is not None else []}
    if train_df is not None:
        for col in ["rgb_path", "video_path", "path", "filepath"]:
            if col in train_df.columns:
                task["video_col"] = col
                break
        for col in ["accident_time", "time", "timestamp", "event_time"]:
            if col in train_df.columns:
                task["time_col"] = col
                break
        if {"center_x", "center_y"}.issubset(train_df.columns):
            task["spatial_cols"] = ["center_x", "center_y"]
        if {"x1", "y1", "x2", "y2"}.issubset(train_df.columns):
            task["bbox_cols"] = ["x1", "y1", "x2", "y2"]
        for col in ["type", "cause", "label", "class", "collision_type"]:
            if col in train_df.columns:
                task["cause_col"] = col
                break
        for col in ["map", "scene_layout", "camera_position", "weather"]:
            if col in train_df.columns:
                task["group_col"] = col
                break
        if task["cause_col"] is not None:
            task["cause_classes"] = sorted(train_df[task["cause_col"]].dropna().astype(str).unique().tolist())
    if task["video_col"] is None and test_df is not None:
        for col in ["path", "video_path", "rgb_path", "filepath"]:
            if col in test_df.columns:
                task["video_col"] = col
                break
    return task


TASK = infer_task_config(train_df, test_df, sample_submission_df)
print(json.dumps(TASK, indent=2))
formulation_lines = []
if TASK["time_col"] is not None:
    formulation_lines.append(f"Time head: regression on `{TASK['time_col']}` normalized by clip duration.")
if TASK["spatial_cols"]:
    formulation_lines.append(f"Spatial head: point regression on {TASK['spatial_cols']}.")
elif TASK["bbox_cols"]:
    formulation_lines.append(f"Spatial head: bounding-box regression on {TASK['bbox_cols']}.")
if TASK["cause_col"] is not None:
    formulation_lines.append(f"Cause head: classification on `{TASK['cause_col']}` with classes {TASK['cause_classes']}.")
print("Final task formulation:")
for line in formulation_lines:
    print("-", line)


In [ ]:
@dataclass
class Config:
    num_frames: int = 16
    image_size: int = 112
    batch_size: int = 4
    num_workers: int = 2
    epochs: int = 5
    lr: float = 1e-4
    weight_decay: float = 1e-4
    dropout: float = 0.3
    grad_clip: float = 1.0
    patience: int = 3
    val_size: float = 0.15
    time_loss_weight: float = 1.0
    spatial_loss_weight: float = 1.0
    cause_loss_weight: float = 1.0
    use_center_sampling: bool = False
    pin_memory: bool = DEVICE.type == "cuda"


CFG = Config()
CFG


## 4. Video Loading Pipeline

Robust loader with backend fallbacks and `(C, T, H, W)` tensors for torchvision video models. Short/corrupt clips are padded by repeating the last frame.


In [ ]:
IMAGENET_MEAN = torch.tensor([0.43216, 0.394666, 0.37645]).view(3, 1, 1, 1)
IMAGENET_STD = torch.tensor([0.22803, 0.22145, 0.216989]).view(3, 1, 1, 1)


def resolve_path(rel_path: str, root: Path) -> Path:
    rel_path = str(rel_path)
    if rel_path.startswith("/"):
        return Path(rel_path)
    candidate = root / rel_path
    if candidate.exists():
        return candidate
    matches = list(root.rglob(Path(rel_path).name))
    if matches:
        return matches[0]
    return candidate


def sample_frame_indices(num_available: int, num_frames: int, center_sampling: bool = False) -> np.ndarray:
    if num_available <= 0:
        return np.zeros(num_frames, dtype=np.int64)
    if num_available == 1:
        return np.zeros(num_frames, dtype=np.int64)
    if center_sampling:
        start = int(num_available * 0.2)
        end = max(start + 1, int(num_available * 0.8))
        indices = np.linspace(start, end - 1, num_frames)
    else:
        indices = np.linspace(0, num_available - 1, num_frames)
    return np.clip(np.round(indices), 0, num_available - 1).astype(np.int64)


def _load_video_opencv(video_path: Path) -> Tuple[List[np.ndarray], float]:
    if cv2 is None:
        raise RuntimeError("OpenCV is unavailable")
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    fps = float(fps) if fps and fps > 0 else 20.0
    frames: List[np.ndarray] = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    cap.release()
    return frames, fps


def _load_video_decord(video_path: Path) -> Tuple[List[np.ndarray], float]:
    if decord is None:
        raise RuntimeError("decord is unavailable")
    vr = decord.VideoReader(str(video_path))
    fps = float(vr.get_avg_fps()) if hasattr(vr, "get_avg_fps") else 20.0
    frames = [frame.asnumpy() for frame in vr]
    return frames, fps


def _load_video_pyav(video_path: Path) -> Tuple[List[np.ndarray], float]:
    if av is None:
        raise RuntimeError("PyAV is unavailable")
    container = av.open(str(video_path))
    stream = container.streams.video[0]
    fps = float(stream.average_rate) if stream.average_rate is not None else 20.0
    frames = [frame.to_ndarray(format="rgb24") for frame in container.decode(stream)]
    container.close()
    return frames, fps


def load_video_frames(video_path: Path) -> Tuple[List[np.ndarray], float, str]:
    errors: List[str] = []
    for name, fn in [("decord", _load_video_decord), ("pyav", _load_video_pyav), ("opencv", _load_video_opencv)]:
        try:
            frames, fps = fn(video_path)
            if len(frames) == 0:
                raise RuntimeError("No frames decoded")
            return frames, fps, name
        except Exception as exc:
            errors.append(f"{name}: {exc}")
    raise RuntimeError(f"All video loaders failed for {video_path}. " + " | ".join(errors))


def preprocess_frames(frames: List[np.ndarray], image_size: int) -> torch.Tensor:
    resize = transforms.Resize((image_size, image_size), interpolation=InterpolationMode.BILINEAR)
    processed: List[torch.Tensor] = []
    for frame in frames:
        tensor = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0
        tensor = resize(tensor)
        processed.append(tensor)
    video = torch.stack(processed, dim=1)
    video = (video - IMAGENET_MEAN) / IMAGENET_STD
    return video


def load_video_tensor(video_path: Path, num_frames: int, image_size: int, center_sampling: bool = False) -> Tuple[torch.Tensor, Dict[str, Any]]:
    frames, fps, backend = load_video_frames(video_path)
    indices = sample_frame_indices(len(frames), num_frames, center_sampling=center_sampling)
    sampled = [frames[i] for i in indices]
    if len(sampled) < num_frames and sampled:
        sampled.extend([sampled[-1]] * (num_frames - len(sampled)))
    video = preprocess_frames(sampled, image_size=image_size)
    meta = {"num_available_frames": len(frames), "sampled_indices": indices.tolist(), "fps": fps, "backend": backend}
    return video, meta


## 5. Dataset and DataLoader

Grouped validation split when no official split is provided. Augmentations stay conservative for surveillance footage.


In [ ]:
def build_splits(train_df: pd.DataFrame, cfg: Config, task: Dict[str, Any]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if len(train_df) < 2:
        return train_df.copy(), train_df.iloc[:0].copy()
    if task["group_col"] is not None and task["group_col"] in train_df.columns:
        groups = train_df[task["group_col"]].astype(str)
    else:
        groups = pd.Series(np.arange(len(train_df)), index=train_df.index)
    splitter = GroupShuffleSplit(n_splits=1, test_size=cfg.val_size, random_state=SEED)
    train_idx, val_idx = next(splitter.split(train_df, groups=groups))
    return train_df.iloc[train_idx].reset_index(drop=True), train_df.iloc[val_idx].reset_index(drop=True)


if train_df is not None:
    train_split_df, valid_split_df = build_splits(train_df, CFG, TASK)
else:
    train_split_df, valid_split_df = None, None

print("train split:", None if train_split_df is None else train_split_df.shape)
print("valid split:", None if valid_split_df is None else valid_split_df.shape)


In [ ]:
class AccidentDataset(Dataset):
    def __init__(self, df: pd.DataFrame, root: Path, cfg: Config, task: Dict[str, Any], mode: str = "train") -> None:
        self.df = df.reset_index(drop=True)
        self.root = root
        self.cfg = cfg
        self.task = task
        self.mode = mode
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(task.get("cause_classes", []))}

    def __len__(self) -> int:
        return len(self.df)

    def _apply_video_aug(self, video: torch.Tensor) -> torch.Tensor:
        if self.mode != "train":
            return video
        if random.random() < 0.2:
            video = torch.clamp(video * (1.0 + random.uniform(-0.05, 0.05)), -4.0, 4.0)
        return video

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        rel_video_path = row[self.task["video_col"]]
        video_path = resolve_path(rel_video_path, self.root)
        try:
            video, meta = load_video_tensor(video_path=video_path, num_frames=self.cfg.num_frames, image_size=self.cfg.image_size, center_sampling=self.cfg.use_center_sampling)
        except Exception as exc:
            warnings.warn(f"Falling back to blank clip for {video_path}: {exc}")
            video = torch.zeros(3, self.cfg.num_frames, self.cfg.image_size, self.cfg.image_size, dtype=torch.float32)
            meta = {"backend": "blank", "fps": 0.0, "num_available_frames": 0, "sampled_indices": []}
        video = self._apply_video_aug(video)
        sample: Dict[str, Any] = {"video": video, "path": str(rel_video_path), "meta": meta}
        duration = float(row["duration"]) if "duration" in row.index and pd.notna(row["duration"]) else np.nan
        sample["duration"] = duration
        if self.task["time_col"] is not None and self.task["time_col"] in row.index and pd.notna(row[self.task["time_col"]]):
            raw_time = float(row[self.task["time_col"]])
            norm_time = raw_time / duration if duration == duration and duration > 0 else raw_time
            sample["time_target"] = torch.tensor(norm_time, dtype=torch.float32)
            sample["time_target_raw"] = torch.tensor(raw_time, dtype=torch.float32)
        if self.task["spatial_cols"] and all(col in row.index for col in self.task["spatial_cols"]):
            sample["spatial_target"] = torch.tensor([float(row[col]) for col in self.task["spatial_cols"]], dtype=torch.float32)
        if self.task["bbox_cols"] and all(col in row.index for col in self.task["bbox_cols"]):
            sample["bbox_target"] = torch.tensor([float(row[col]) for col in self.task["bbox_cols"]], dtype=torch.float32)
        if self.task["cause_col"] is not None and self.task["cause_col"] in row.index and pd.notna(row[self.task["cause_col"]]):
            sample["cause_target"] = torch.tensor(self.class_to_idx[str(row[self.task["cause_col"]])], dtype=torch.long)
        return sample


def accident_collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    out: Dict[str, Any] = {"video": torch.stack([item["video"] for item in batch]), "path": [item["path"] for item in batch], "meta": [item["meta"] for item in batch], "duration": torch.tensor([item.get("duration", np.nan) for item in batch], dtype=torch.float32)}
    for key in ["time_target", "time_target_raw", "spatial_target", "bbox_target", "cause_target"]:
        values = [item[key] for item in batch if key in item]
        if len(values) == len(batch):
            out[key] = torch.stack(values)
    return out


train_loader = valid_loader = test_loader = None
if train_split_df is not None and len(train_split_df) > 0:
    train_ds = AccidentDataset(train_split_df, COMP_ROOT, CFG, TASK, mode="train")
    valid_ds = AccidentDataset(valid_split_df, COMP_ROOT, CFG, TASK, mode="valid")
    train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers, pin_memory=CFG.pin_memory, collate_fn=accident_collate_fn)
    valid_loader = DataLoader(valid_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=CFG.pin_memory, collate_fn=accident_collate_fn)

if test_df is not None and len(test_df) > 0:
    test_ds = AccidentDataset(test_df, COMP_ROOT, CFG, TASK, mode="test")
    test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=CFG.pin_memory, collate_fn=accident_collate_fn)

print("train_loader ready:", train_loader is not None)
print("valid_loader ready:", valid_loader is not None)
print("test_loader ready:", test_loader is not None)


## 6. Baseline Model

R(2+1)D backbone with task-specific heads. Each head is created only if its target exists.


In [ ]:
def load_r2plus1d_backbone() -> nn.Module:
    try:
        from torchvision.models.video import R2Plus1D_18_Weights
        model = r2plus1d_18(weights=R2Plus1D_18_Weights.DEFAULT)
        print("Loaded pretrained R(2+1)D weights.")
        return model
    except Exception as exc:
        warnings.warn(f"Falling back to randomly initialized R(2+1)D: {exc}")
        return r2plus1d_18(weights=None)


class MultiHeadVideoModel(nn.Module):
    def __init__(self, task: Dict[str, Any], dropout: float = 0.3) -> None:
        super().__init__()
        self.task = task
        backbone = load_r2plus1d_backbone()
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.dropout = nn.Dropout(dropout)
        self.time_head = nn.Linear(in_features, 1) if task["time_col"] is not None else None
        if task["spatial_cols"]:
            self.spatial_head = nn.Linear(in_features, len(task["spatial_cols"]))
        elif task["bbox_cols"]:
            self.spatial_head = nn.Linear(in_features, len(task["bbox_cols"]))
        else:
            self.spatial_head = None
        self.cause_head = nn.Linear(in_features, len(task["cause_classes"])) if task["cause_classes"] else None

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        feats = self.backbone(x)
        feats = self.dropout(feats)
        out: Dict[str, torch.Tensor] = {}
        if self.time_head is not None:
            out["time"] = self.time_head(feats).squeeze(1)
        if self.spatial_head is not None:
            out["spatial"] = self.spatial_head(feats)
        if self.cause_head is not None:
            out["cause"] = self.cause_head(feats)
        return out


model = MultiHeadVideoModel(TASK, dropout=CFG.dropout).to(DEVICE)
model


## 7. Losses and Metrics (Proxy)

Proxy metrics only; the competition metric is private. Track per-head losses plus MAE for time/spatial and accuracy for cause.


In [ ]:
def compute_losses(outputs: Dict[str, torch.Tensor], batch: Dict[str, torch.Tensor], cfg: Config) -> Tuple[torch.Tensor, Dict[str, float]]:
    total_loss = torch.tensor(0.0, device=DEVICE)
    logs: Dict[str, float] = {}
    if "time" in outputs and "time_target" in batch:
        loss_time = F.smooth_l1_loss(outputs["time"], batch["time_target"].to(DEVICE))
        total_loss = total_loss + cfg.time_loss_weight * loss_time
        logs["loss_time"] = float(loss_time.detach().cpu())
    if "spatial" in outputs:
        target_key = "spatial_target" if "spatial_target" in batch else "bbox_target" if "bbox_target" in batch else None
        if target_key is not None:
            loss_spatial = F.smooth_l1_loss(outputs["spatial"], batch[target_key].to(DEVICE))
            total_loss = total_loss + cfg.spatial_loss_weight * loss_spatial
            logs["loss_spatial"] = float(loss_spatial.detach().cpu())
    if "cause" in outputs and "cause_target" in batch:
        loss_cause = F.cross_entropy(outputs["cause"], batch["cause_target"].to(DEVICE))
        total_loss = total_loss + cfg.cause_loss_weight * loss_cause
        logs["loss_cause"] = float(loss_cause.detach().cpu())
    logs["loss_total"] = float(total_loss.detach().cpu())
    return total_loss, logs


@torch.no_grad()
def evaluate(model: nn.Module, loader: Optional[DataLoader], cfg: Config) -> Tuple[Dict[str, float], Optional[pd.DataFrame]]:
    if loader is None:
        return {}, None
    model.eval()
    losses: List[Dict[str, float]] = []
    rows: List[Dict[str, Any]] = []
    for batch in loader:
        videos = batch["video"].to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            outputs = model(videos)
            _, loss_log = compute_losses(outputs, batch, cfg)
        losses.append(loss_log)
        for i, path in enumerate(batch["path"]):
            row: Dict[str, Any] = {"path": path}
            duration = float(batch["duration"][i].item()) if not torch.isnan(batch["duration"][i]) else np.nan
            row["duration"] = duration
            if "time" in outputs:
                pred_time_norm = float(outputs["time"][i].detach().cpu())
                row["pred_time_norm"] = pred_time_norm
                row["pred_time"] = pred_time_norm * duration if duration == duration and duration > 0 else pred_time_norm
            if "time_target_raw" in batch:
                row["true_time"] = float(batch["time_target_raw"][i].item())
            if "spatial" in outputs:
                spatial_pred = outputs["spatial"][i].detach().cpu().numpy().tolist()
                for j, value in enumerate(spatial_pred):
                    row[f"pred_spatial_{j}"] = value
            if "spatial_target" in batch:
                spatial_true = batch["spatial_target"][i].detach().cpu().numpy().tolist()
                for j, value in enumerate(spatial_true):
                    row[f"true_spatial_{j}"] = value
            if "cause" in outputs:
                pred_idx = int(torch.argmax(outputs["cause"][i]).item())
                row["pred_cause_idx"] = pred_idx
                row["pred_cause"] = TASK["cause_classes"][pred_idx]
            if "cause_target" in batch:
                true_idx = int(batch["cause_target"][i].item())
                row["true_cause_idx"] = true_idx
                row["true_cause"] = TASK["cause_classes"][true_idx]
            rows.append(row)
    metrics = {key: float(np.mean([x[key] for x in losses if key in x])) for key in losses[0].keys()} if losses else {}
    pred_df = pd.DataFrame(rows) if rows else None
    if pred_df is not None and {"true_time", "pred_time"}.issubset(pred_df.columns):
        metrics["time_mae_proxy"] = float(np.mean(np.abs(pred_df["pred_time"] - pred_df["true_time"])))
    if pred_df is not None and {"true_spatial_0", "pred_spatial_0", "true_spatial_1", "pred_spatial_1"}.issubset(pred_df.columns):
        pred_xy = pred_df[["pred_spatial_0", "pred_spatial_1"]].values
        true_xy = pred_df[["true_spatial_0", "true_spatial_1"]].values
        metrics["spatial_mae_proxy"] = float(np.mean(np.abs(pred_xy - true_xy)))
    if pred_df is not None and {"true_cause_idx", "pred_cause_idx"}.issubset(pred_df.columns):
        metrics["cause_acc_proxy"] = float((pred_df["true_cause_idx"] == pred_df["pred_cause_idx"]).mean())
    proxy_terms: List[float] = []
    if "time_mae_proxy" in metrics:
        proxy_terms.append(1.0 / (1.0 + metrics["time_mae_proxy"]))
    if "spatial_mae_proxy" in metrics:
        proxy_terms.append(1.0 / (1.0 + metrics["spatial_mae_proxy"]))
    if "cause_acc_proxy" in metrics:
        proxy_terms.append(metrics["cause_acc_proxy"])
    if proxy_terms:
        metrics["composite_proxy"] = float(np.mean(proxy_terms))
    return metrics, pred_df


## 8. Training Loop

AMP, grad scaling, grad clipping, cosine LR, early stopping, checkpointing to working directory.


In [ ]:
def train_model(model: nn.Module, train_loader: Optional[DataLoader], valid_loader: Optional[DataLoader], cfg: Config) -> Tuple[nn.Module, pd.DataFrame]:
    if train_loader is None:
        warnings.warn("Training loader is unavailable. Skipping training.")
        return model, pd.DataFrame()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(cfg.epochs, 1))
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    best_score = -np.inf
    best_state = None
    wait = 0
    history: List[Dict[str, float]] = []
    for epoch in range(1, cfg.epochs + 1):
        model.train()
        epoch_logs: List[Dict[str, float]] = []
        for batch in train_loader:
            videos = batch["video"].to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                outputs = model(videos)
                loss, loss_log = compute_losses(outputs, batch, cfg)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            epoch_logs.append(loss_log)
        scheduler.step()
        train_metrics = {f"train_{k}": float(np.mean([x[k] for x in epoch_logs if k in x])) for k in epoch_logs[0].keys()} if epoch_logs else {}
        valid_metrics, _ = evaluate(model, valid_loader, cfg)
        valid_metrics = {f"valid_{k}": v for k, v in valid_metrics.items()}
        row = {"epoch": epoch, **train_metrics, **valid_metrics, "lr": optimizer.param_groups[0]["lr"]}
        history.append(row)
        tracked_score = row.get("valid_composite_proxy", -row.get("valid_loss_total", np.inf))
        if tracked_score > best_score:
            best_score = tracked_score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_state, WORKING_ROOT / "best_r2plus1d_baseline.pt")
            wait = 0
        else:
            wait += 1
        print(f"Epoch {epoch:02d} | train loss: {row.get('train_loss_total', np.nan):.4f} | valid loss: {row.get('valid_loss_total', np.nan):.4f} | valid composite proxy: {row.get('valid_composite_proxy', np.nan):.4f}")
        print({k: round(v, 4) for k, v in row.items() if k not in {'epoch', 'lr'}})
        if wait >= cfg.patience:
            print(f"Early stopping triggered at epoch {epoch}.")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


model, history_df = train_model(model, train_loader, valid_loader, CFG)
display(history_df.tail())


## 9. Validation and Error Analysis


In [ ]:
valid_metrics, valid_pred_df = evaluate(model, valid_loader, CFG)
print(valid_metrics)
if valid_pred_df is not None:
    display(valid_pred_df.head())


In [ ]:
if valid_pred_df is not None and {"true_time", "pred_time"}.issubset(valid_pred_df.columns):
    plt.figure(figsize=(6, 6))
    plt.scatter(valid_pred_df["true_time"], valid_pred_df["pred_time"], alpha=0.5)
    lim = max(valid_pred_df["true_time"].max(), valid_pred_df["pred_time"].max())
    plt.plot([0, lim], [0, lim], "r--")
    plt.xlabel("True accident time")
    plt.ylabel("Predicted accident time")
    plt.title("Time Regression Proxy")
    plt.show()

if valid_pred_df is not None and {"true_spatial_0", "pred_spatial_0", "true_spatial_1", "pred_spatial_1"}.issubset(valid_pred_df.columns):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(valid_pred_df["true_spatial_0"], valid_pred_df["pred_spatial_0"], alpha=0.5)
    axes[0].plot([0, 1], [0, 1], "r--")
    axes[0].set_title("center_x")
    axes[1].scatter(valid_pred_df["true_spatial_1"], valid_pred_df["pred_spatial_1"], alpha=0.5)
    axes[1].plot([0, 1], [0, 1], "r--")
    axes[1].set_title("center_y")
    plt.show()

if valid_pred_df is not None and {"true_cause", "pred_cause"}.issubset(valid_pred_df.columns):
    labels = TASK["cause_classes"]
    cm = confusion_matrix(valid_pred_df["true_cause"], valid_pred_df["pred_cause"], labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
    plt.title("Cause Classification Proxy")
    plt.show()


In [ ]:
if valid_pred_df is not None and {"true_time", "pred_time"}.issubset(valid_pred_df.columns):
    valid_pred_df["time_abs_error"] = (valid_pred_df["pred_time"] - valid_pred_df["true_time"]).abs()
    display(valid_pred_df.sort_values("time_abs_error", ascending=False).head(10))
    print("Likely failure modes: domain shift, low image quality, occlusion, ambiguous onset.")


## 10. Test Inference and Submission


In [ ]:
@torch.no_grad()
def predict_test(model: nn.Module, loader: Optional[DataLoader]) -> pd.DataFrame:
    if loader is None:
        warnings.warn("Test loader is unavailable.")
        return pd.DataFrame()
    model.eval()
    rows: List[Dict[str, Any]] = []
    default_type = TASK["cause_classes"][0] if TASK["cause_classes"] else "rear-end"
    for batch in loader:
        videos = batch["video"].to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            outputs = model(videos)
        for i, path in enumerate(batch["path"]):
            duration = float(batch["duration"][i].item()) if not torch.isnan(batch["duration"][i]) else np.nan
            row = {"path": path}
            if "time" in outputs:
                pred_time = float(outputs["time"][i].detach().cpu())
                if duration == duration and duration > 0:
                    pred_time = float(np.clip(pred_time * duration, 0.0, duration))
                row["accident_time"] = pred_time
            else:
                row["accident_time"] = float(duration * 0.5) if duration == duration and duration > 0 else 0.0
            if "spatial" in outputs and outputs["spatial"].shape[1] >= 2:
                spatial = outputs["spatial"][i].detach().cpu().numpy()
                row["center_x"] = float(np.clip(spatial[0], 0.0, 1.0))
                row["center_y"] = float(np.clip(spatial[1], 0.0, 1.0))
            else:
                row["center_x"] = 0.5
                row["center_y"] = 0.5
            if "cause" in outputs:
                pred_idx = int(torch.argmax(outputs["cause"][i]).item())
                row["type"] = TASK["cause_classes"][pred_idx]
            else:
                row["type"] = default_type
            rows.append(row)
    return pd.DataFrame(rows)


test_pred_df = predict_test(model, test_loader)
display(test_pred_df.head())


In [ ]:
def build_submission(test_pred_df: pd.DataFrame, sample_submission_df: Optional[pd.DataFrame], test_df: Optional[pd.DataFrame]) -> pd.DataFrame:
    if sample_submission_df is not None and len(sample_submission_df) > 0:
        submission = sample_submission_df.copy()
    elif test_df is not None:
        submission = pd.DataFrame({"path": test_df[TASK["video_col"]].astype(str)})
        for col in ["accident_time", "center_x", "center_y", "type"]:
            submission[col] = np.nan
    else:
        raise RuntimeError("No sample submission or test metadata was found.")
    merged = submission.drop(columns=[c for c in submission.columns if c in test_pred_df.columns and c != "path"], errors="ignore").merge(test_pred_df, on="path", how="left")
    for col in submission.columns:
        if col not in merged.columns:
            if col == "accident_time":
                merged[col] = 0.0
            elif col in {"center_x", "center_y"}:
                merged[col] = 0.5
            elif col == "type":
                merged[col] = TASK["cause_classes"][0] if TASK["cause_classes"] else "rear-end"
            else:
                merged[col] = submission[col]
    merged = merged[submission.columns]
    return merged


submission_df = build_submission(test_pred_df, sample_submission_df, test_df)
print(submission_df.dtypes)
display(submission_df.head())

submission_path = WORKING_ROOT / "submission.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Saved submission to: {submission_path}")


## 11. Notebook Quality Notes

- 16 frames, 112×112, R(2+1)D baseline
- schema-aware heads and validation split
- safe fallbacks for missing files/backends


## Next Improvements

1. SlowFast swap with the same dataset/training interfaces.
2. Better temporal sampling, including accident-focused or multi-clip inference.
3. Optical flow auxiliary branch fused with RGB video features.
4. Ensembling across temporal crops, seeds, and backbones.
5. Pseudo-labeling or zero-shot adaptation ideas, if allowed by competition rules.
